# 21 - Python 3.12 New Features


## 1. Type Parameter Syntax & `type` Statement (PEP 695)

Python 3.12 ile genel (generic) fonksiyonlar, sınıflar ve tip takma adları (type aliases) tanımlamak için `TypeVar` ve `Generic` içe aktarma zorunluluğunu ortadan kaldıran köşeli parantezli (`[T]`) sözdizimi ile `type` anahtar kelimesi eklenmiştir.

**Syntax**

```Python
type Point[T] = tuple[T, T]

def first_item[T](items: list[T]) -> T:
    return items[0]

class Box[T]:
    def __init__(self, value: T) -> None:
        self.value = value
```

**Temel Özellikler ve Avantajlar**

* **Kapsamlı Tip Parametreleri (`[T]`):** Fonksiyon adından veya sınıf adından hemen sonra yazılan `[T]`, `[T: int | float]` (üst sınır / bound) veya `[T: (int, str)]` (kısıt / constraint) sözdizimi yerel bir kapsam oluşturur; dışarıdaki değişkenlerle isim çakışması yaşanmaz.
* **`type` Deyimi ile Tembel (Lazy) Değerlendirme:** `type Node = list['Node']` gibi özyinelemeli (recursive) tip takma adları tırnak işaretine gerek kalmadan `type Node = list[Node]` biçiminde doğrudan yazılabilir; çalışma zamanında `TypeAliasType` nesnesi üretilir.


In [ ]:
type Point[T: (int, float)] = tuple[T, T]
type Tree[T] = T | list[Tree[T]]


def get_first[T](items: list[T]) -> T:
    return items[0]


class Container[T]:
    def __init__(self, item: T) -> None:
        self.item = item

    def unwrap(self) -> T:
        return self.item


origin: Point[float] = (0.0, 0.0)
nested_tree: Tree[int] = [1, [2, [3, 4]]]
box = Container('Python 3.12')

print('Origin:', origin)
print('First item:', get_first([10, 20, 30]))
print('Container:', box.unwrap())
print('Type alias object:', type(Point))


## 2. Syntactic Formalization of F-Strings (PEP 701)

Python 3.12 ile f-string ayrıştırıcısı doğrudan PEG dilbilgisine (grammar) entegre edilerek eski sürüm kısıtlamaları kaldırılmış; iç ifadelerde aynı tırnak işaretlerini yeniden kullanma, çok satırlı ifadeler, satır içi yorumlar ve ters eğik çizgi (`\`) kullanımı serbest bırakılmıştır.

**Syntax**

```Python
songs = ['Alkaline', 'Ascensionism']
text = f"Playlist: {', '.join(songs)}"
```

**Kaldırılan Kısıtlamalar**

* **Aynı Tırnak İşaretini Yeniden Kullanma:** Çift tırnakla başlayan bir f-string içinde (`f"..."`) süslü parantezlerin içine kaçış karakteri kullanmadan doğrudan çift tırnak (`"..."`) yazılabilir.
* **Çok Satırlı İfadeler ve Yorumlar:** Süslü parantez (`{...}`) içindeki karmaşık ifadeler okunabilirlik için alt satırlara bölünebilir ve her satırın yanına `#` ile yorum eklenebilir.
* **Ters Eğik Çizgi (`\`) ve Sınırsız İç İçe Geçme (Nesting):** İfade blokları içinde `\n` veya `\t` gibi kaçış dizileri doğrudan kullanılabilir; f-string ifadeleri istenilen derinlikte iç içe yerleştirilebilir.


In [ ]:
tracks: list[str] = ['El Mirador', 'Entre Dos Aguas', 'Zyryab']

# Aynı çift tırnağı kaçışsız kullanma ve çok satırlı + yorumlu ifade
summary: str = f"Setlist ({len(tracks)} parça): {
    ' | '.join(track.upper() for track in tracks)  # tüm parçaları büyük harfe çevir
}"
print(summary)

# F-string ifadesi içinde doğrudan newline kaçışı kullanımı
multiline_block: str = f"Parçalar:\n{'\n'.join(f'- {t}' for t in tracks)}"
print(multiline_block)


## 3. `@typing.override` Decorator (PEP 698)

`@typing.override` dekoratörü, bir alt sınıftaki metodun üst sınıftaki (veya arayüzdeki) bir metodu bilerek ezdiğini (override) açıkça işaretler; üst sınıftaki metodun adı değiştiğinde veya silindiğinde statik tip denetleyicilerin (Pyright, Mypy) hatayı anında yakalamasını sağlar.

**Syntax**

```python
from typing import override


class Child(Parent):
    @override
    def run(self) -> None:
        ...
```

**Kullanım Kuralları**

* **Sessiz Hataları Önleme:** Üst sınıfta `process_audio()` olan metot `render_audio()` olarak yeniden adlandırıldığında, `@override` kullanılmadıysa alt sınıftaki eski `process_audio()` fark edilmeden ölü kod olarak kalır; `@override` bu uyumsuzluğu derleme öncesi tespit eder.
* **Çalışma Zamanı Davranışı:** Dekoratör metot üzerinde `__override__ = True` özniteliğini ayarlar ve çalışma zamanında ek performans maliyeti oluşturmaz.


In [ ]:
from typing import override


class BaseProcessor:
    def execute(self) -> str:
        return 'Base execution'


class AudioProcessor(BaseProcessor):
    @override
    def execute(self) -> str:
        return 'Audio DSP execution'


proc = AudioProcessor()
print(proc.execute())
print('Has __override__ marker:', getattr(AudioProcessor.execute, '__override__', False))


## 4. Python Buffer Protocol (`collections.abc.Buffer` & `__buffer__`) (PEP 688)

Python 3.12 öncesinde yalnızca C uzantılarından erişilebilen bellek tamponu protokolü (Buffer Protocol), `__buffer__` ve `__release_buffer__` özel metotları ile saf Python sınıflarına açılmış ve tip ek açıklamaları için `collections.abc.Buffer` soyut temel sınıfı eklenmiştir.

**Syntax**

```python
from collections.abc import Buffer
from inspect import BufferFlags


class CustomBuffer:
    def __buffer__(self, flags: int) -> memoryview:
        ...
```

**Kullanım Alanları**

* **Sıfır Kopyalı Bellek Erişimi (Zero-Copy):** `memoryview()`, `bytes()` veya NumPy/OpenCV gibi kütüphanelere aktarılabilecek özel veri yapıları doğrudan Python içinde yazılabilir.
* **Doğru Tip İpuçları:** `bytes | bytearray | memoryview` gibi uzun birleşimler yerine `data: Buffer` yazılarak tampon protokolünü destekleyen tüm yerleşik ve özel nesneler tek tiple kabul edilir.


In [ ]:
from collections.abc import Buffer


class PacketBuffer:
    def __init__(self, payload: bytes) -> None:
        self._data = bytearray(payload)

    def __buffer__(self, flags: int) -> memoryview:
        return memoryview(self._data)

    def __release_buffer__(self, view: memoryview) -> None:
        view.release()


def read_header(buf: Buffer) -> list[int]:
    with memoryview(buf) as view:
        return list(view[:4])


packet = PacketBuffer(b'\x89PNG\r\n\x1a\n')
print('isinstance Buffer:', isinstance(packet, Buffer))
print('Header bytes:', read_header(packet))


## 5. Comprehension Inlining (PEP 709) & Low-Impact Monitoring (`sys.monitoring`) (PEP 669)

Python 3.12 çalışma zamanı motorunda liste/sözlük/küme üreteçlerinin (comprehensions) derlenme biçimi değiştirilmiş ve hata ayıklayıcılar (debuggers) ile kod kapsama (coverage) araçları için neredeyse sıfır maliyetli olay izleme API'si sunulmuştur.

**Syntax**

```python
import sys

TOOL_ID = sys.monitoring.DEBUGGER_ID
sys.monitoring.use_tool_id(TOOL_ID, 'custom_tracer')
```

**Mimari İyileştirmeler**

* **Comprehension Inlining (PEP 709):** `[x * 2 for x in data]` gibi ifadeler artık arka planda gizli bir fonksiyon nesnesi oluşturup her çağrıda yeni bir yığın çerçevesi (stack frame) açmaz; doğrudan bulundukları fonksiyonun bytecode'u içine gömülerek çalışır ve yaklaşık 2 kata kadar hız artışı sağlar.
* **`sys.monitoring` (PEP 669):** Eski `sys.settrace()` mekanizması her satırda kanca tetiklediği için kodu ciddi oranda yavaşlatırken, `sys.monitoring` yalnızca abone olunan olayları (`PY_START`, `LINE`, `RAISE` vb.) bytecode düzeyinde etkinleştirerek profilleme ve hata ayıklama maliyetini minimuma indirir.
* **Gelişmiş Hata Mesajları:** `NameError`, `ImportError` ve `SyntaxError` durumlarında standart kütüphane modülleri, sınıf üyeleri ve `import x from y` gibi ters yazılmış ifadeler için doğrudan düzeltme önerileri (`Did you mean...`) üretilir.


In [ ]:
import sys
import timeit

# PEP 709: Inlined list comprehension hız testi
comp_time: float = timeit.timeit('[x * x for x in range(1_000)]', number=20_000)
print(f'Inlined comprehension (20k runs): {comp_time:.4f}s')

# PEP 669: sys.monitoring ile hafif olay izleme örneği
TOOL_ID: int = sys.monitoring.OPTIMIZER_ID
events_logged: list[str] = []

sys.monitoring.use_tool_id(TOOL_ID, 'demo_monitor')


def on_call(code: object, instruction_offset: int, callable_obj: object, arg0: object) -> object:
    name = getattr(callable_obj, '__name__', str(callable_obj))
    if name == 'sample_worker':
        events_logged.append(f'CALL -> {name} @ offset {instruction_offset}')
    return sys.monitoring.DISABLE


def sample_worker(val: int) -> int:
    return val + 1


sys.monitoring.register_callback(TOOL_ID, sys.monitoring.events.CALL, on_call)
sys.monitoring.set_events(TOOL_ID, sys.monitoring.events.CALL)

sample_worker(41)

sys.monitoring.set_events(TOOL_ID, 0)
sys.monitoring.free_tool_id(TOOL_ID)
print('Monitored events:', events_logged)
